In [28]:
import polars as pl
import os
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, auc

In [6]:
DATASET_PATH = '../dataset/E066/'

In [32]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [7]:
# Load dataset
E066_pl = pl.read_parquet(os.path.join(DATASET_PATH, 'E066_w_histone_pl.parquet'))

In [15]:
perm_pl = pl.read_parquet('permutation.parquet')
perm_lst = perm_pl['markers_perm'].to_list()

In [77]:
item = perm_lst[100]
item_name = '-'.join(item)
print(item)
print(item_name)

['H3K36me3', 'H3K4me1', 'H3K9me3', 'H3K4me3']
H3K36me3-H3K4me1-H3K9me3-H3K4me3


In [78]:
# Create dataframe based on histone marker permutation item

perm_histone = E066_pl.with_columns(
    pl.struct(item).map_elements(
        lambda x: [x[col_name] for col_name in item],
        return_dtype = pl.List(pl.List(pl.Float64))
    )
    .alias('histone')
)

In [79]:
perm_histone.head(2)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K36me3,H3K36me3_wc,H3K36me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E066,label,all_histone,histone
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,i64,list[list[f64]],list[list[f64]]
"""ENSG00000000003""","[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",9,100,"[3.39243, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,73.205,1,"[[0.0, 0.0, … 0.0], [0.0, 4.19024, … 0.0], … [3.39243, 0.0, … 0.0]]","[[3.39243, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 4.19024, … 0.0]]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,0.191,0,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"


In [80]:
# Select X and y column
X = perm_histone.select(pl.col('histone')).to_series().to_list()
y = perm_histone.select(pl.col('label')).to_series().to_list()

In [81]:
# Convert to Numpy array
X = np.array(X)
y = np.array(y)

In [82]:
# Split the dataset into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.666, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [83]:
print("Train set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

Train set shape: (6561, 4, 100) (6561,)
Validation set shape: (6542, 4, 100) (6542,)
Test set shape: (6542, 4, 100) (6542,)


In [84]:
# Convert numpy arrays to PyTorch tensors

X_train = torch.from_numpy(X_train).float().unsqueeze(1).to(device)
X_val = torch.from_numpy(X_val).float().unsqueeze(1).to(device)
X_test = torch.from_numpy(X_test).float().unsqueeze(1).to(device)

y_train = torch.from_numpy(y_train).float().to(device)
y_val = torch.from_numpy(y_val).float().to(device)
y_test = torch.from_numpy(y_test).float().to(device)

In [85]:
# Create DataLoaders
batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [86]:
class DeepClassifier(nn.Module):
    def __init__(self, input_dim = [5, 100], out_channels = 50, conv_kernel_size = 10, max_pooling_kernel_size = 5):
        super(DeepClassifier, self).__init__()
        
        flatten_dim = out_channels * ((input_dim[1] - conv_kernel_size + 1) // max_pooling_kernel_size)

        self.conv = nn.Conv2d(1, out_channels, kernel_size=(input_dim[0], conv_kernel_size))
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d((1, max_pooling_kernel_size))
        self.dropout = nn.Dropout(0.5)
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(flatten_dim, 625)
        self.linear2 = nn.Linear(625, 125)
        self.linear3 = nn.Linear(125, 2)
        self.softmax = nn.LogSoftmax(dim=1)
        

    def forward(self, x):
        # Stage 1: filter bank -> squashing -> max pooling
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)

        # Stage 2: standar 2-layer neural network        
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.relu(x)
        x = self.linear3(x)
        x = self.softmax(x)
        return x

In [87]:
n_rows_item = len(item)
print(n_rows_item)

4


In [88]:
model = DeepClassifier(input_dim = [n_rows_item, 100]).to(device)
criterion = nn.NLLLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = optim.SGD(model.parameters(), lr=0.001)
num_epochs = 100

# For tracking train and validation
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
train_aucs, val_aucs = [], []

# For saving the best model
best_val_metric = float('-inf')
best_model_path = f'{item_name}.pth'

In [110]:
output_df = pd.DataFrame(columns=[
    "train_loss_min", "train_loss_avg", "train_loss_max",
    "train_acc_min", "train_acc_avg", "train_acc_max",
    "train_auc_min", "train_auc_avg", "train_auc_max",
    "val_loss_min", "val_loss_avg", "val_loss_max",
    "val_acc_min", "val_acc_avg", "val_acc_max",
    "val_auc_min", "val_auc_avg", "val_auc_max",
])

In [89]:
for epoch in range(num_epochs):
    # Training phase
    model.train()  # Set the model to training mode
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    train_true_labels = []
    train_predicted_probs = []
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.cuda(), labels.type(torch.LongTensor).cuda())
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

        train_true_labels.extend(labels.cpu().numpy())
        train_predicted_probs.extend(predicted.cpu().numpy())
    
    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = 100 * train_correct / train_total
    train_auc_score = roc_auc_score(train_true_labels, train_predicted_probs)
    
    train_losses.append(round(avg_train_loss, 4))
    train_accuracies.append(round(train_accuracy, 2))
    train_aucs.append(round(train_auc_score, 2))
    
    # Validation phase
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_metric = 0
    val_true_labels = []
    val_predicted_probs = []
    
    with torch.no_grad():  # Disable gradient computation
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs.cuda(), labels.type(torch.LongTensor).cuda())
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            val_true_labels.extend(labels.cpu().numpy())
            val_predicted_probs.extend(predicted.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total
    val_metric = val_correct / len(val_loader)

    # Save the best model
    if val_metric > best_val_metric:
        best_val_metric = val_metric
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved with validation metric: {best_val_metric:.4f}")
    
    val_true_labels = np.array(val_true_labels)
    val_predicted_probs = np.array(val_predicted_probs)
    val_auc_score = roc_auc_score(val_true_labels, val_predicted_probs)

    val_losses.append(round(avg_val_loss, 4))
    val_accuracies.append(round(val_accuracy, 2))
    val_aucs.append(round(val_auc_score, 2))
    
    print(f'Epoch [{epoch+1}/{num_epochs}] - '
          f'[TRAIN] Loss: {avg_train_loss:.4f}, Accuracy: {train_accuracy:.2f}%, AUC: {train_auc_score:.2f} - '
          f'[VAL] Loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.2f}%, AUC: {val_auc_score:.2f}')
print("Training finished!")

New best model saved with validation metric: 23.5122
Epoch [1/100] - [TRAIN] Loss: 0.6132, Accuracy: 69.30%, AUC: 0.69 - [VAL] Loss: 0.5950, Accuracy: 73.68%, AUC: 0.74
New best model saved with validation metric: 24.4439
Epoch [2/100] - [TRAIN] Loss: 0.5840, Accuracy: 75.40%, AUC: 0.75 - [VAL] Loss: 0.5765, Accuracy: 76.60%, AUC: 0.77
New best model saved with validation metric: 24.9073
Epoch [3/100] - [TRAIN] Loss: 0.5682, Accuracy: 78.04%, AUC: 0.78 - [VAL] Loss: 0.5600, Accuracy: 78.05%, AUC: 0.78
New best model saved with validation metric: 25.0732
Epoch [4/100] - [TRAIN] Loss: 0.5561, Accuracy: 79.09%, AUC: 0.79 - [VAL] Loss: 0.5495, Accuracy: 78.57%, AUC: 0.79
Epoch [5/100] - [TRAIN] Loss: 0.5470, Accuracy: 79.27%, AUC: 0.79 - [VAL] Loss: 0.5545, Accuracy: 77.93%, AUC: 0.78
Epoch [6/100] - [TRAIN] Loss: 0.5365, Accuracy: 79.13%, AUC: 0.79 - [VAL] Loss: 0.5452, Accuracy: 78.31%, AUC: 0.78
New best model saved with validation metric: 25.1268
Epoch [7/100] - [TRAIN] Loss: 0.5286, A

In [101]:
# Load the best model
best_model = DeepClassifier(input_dim = [n_rows_item, 100]).to(device)
best_model.load_state_dict(torch.load(best_model_path))
best_model.eval()

# Predict on test set
test_predictions = []
test_true_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = best_model(inputs)
        # probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        test_predictions.extend(predicted.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())

# Calculate evaluation metric
test_acc_score = round(accuracy_score(test_true_labels, test_predictions) * 100, 2)
print(f"Accuracy Score on test set: {test_acc_score:.2f} %")
test_auc_score = round(roc_auc_score(test_true_labels, test_predictions), 2)
print(f"AUC Score on test set: {test_auc_score:.2f}")

Accuracy Score on test set: 80.63 %
AUC Score on test set: 0.81


In [93]:
# Create dataframe from array
experiment_results = {
    'train_loss': train_losses,
    'train_accuracy': train_accuracies,
    'train_auc': train_aucs,
    'val_loss': val_losses,
    'val_accuracy': val_accuracies,
    'val_auc': val_aucs
}

In [94]:
experiment_df = pl.DataFrame(experiment_results)

In [95]:
experiment_df.write_csv(os.path.join(DATASET_PATH, 'experiments', f'{item_name}.csv'))

In [96]:
def min_avg_max(lst):
    return round(min(lst), 2), round(sum(lst)/len(lst), 2), round(max(lst), 2)

In [97]:
train_loss_min, train_loss_avg, train_loss_max = min_avg_max(train_losses)
train_acc_min, train_acc_avg, train_acc_max = min_avg_max(train_accuracies)
train_auc_min, train_auc_avg, train_auc_max = min_avg_max(train_aucs)

print(f"{train_loss_min:.2f}, {train_loss_avg:.2f}, {train_loss_max:.2f}")
print(f"{train_acc_min}, {train_acc_avg}, {train_acc_max}")
print(f"{train_auc_min}, {train_auc_avg}, {train_auc_max}")

0.44, 0.47, 0.61
69.3, 80.15, 81.28
0.69, 0.8, 0.81


In [98]:
val_loss_min, val_loss_avg, val_loss_max = min_avg_max(val_losses)
val_acc_min, val_acc_avg, val_acc_max = min_avg_max(val_accuracies)
val_auc_min, val_auc_avg, val_auc_max = min_avg_max(val_aucs)

print(f"{val_loss_min:.2f}, {val_loss_avg:.2f}, {val_loss_max:.2f}")
print(f"{val_acc_min}, {val_acc_avg}, {val_acc_max}")
print(f"{val_auc_min}, {val_auc_avg}, {val_auc_max}")

0.47, 0.50, 0.59
73.68, 78.86, 79.3
0.74, 0.79, 0.79


In [102]:
print(f"{test_acc_score}, {test_auc_score}")

80.63, 0.81


In [103]:
output_rows = []

In [112]:
output_rows.append([
    train_loss_min, train_loss_avg, train_loss_max,
    train_acc_min, train_acc_avg, train_acc_max,
    train_auc_min, train_auc_avg, train_auc_max,
    val_loss_min, val_loss_avg, val_loss_max,
    val_acc_min, val_acc_avg, val_acc_max,
    val_auc_min, val_auc_avg, val_auc_max,
    
])

In [114]:
output_rows_df = pd.DataFrame(output_rows, 
    columns=[
        "train_loss_min", "train_loss_avg", "train_loss_max",
        "train_acc_min", "train_acc_avg", "train_acc_max",
        "train_auc_min", "train_auc_avg", "train_auc_max",
        "val_loss_min", "val_loss_avg", "val_loss_max",
        "val_acc_min", "val_acc_avg", "val_acc_max",
        "val_auc_min", "val_auc_avg", "val_auc_max",
    ]
)

In [123]:
output_df = pd.concat([output_df, output_rows_df], ignore_index=True)

In [124]:
output_df

,train_loss_min,train_loss_avg,train_loss_max,train_acc_min,train_acc_avg,train_acc_max,train_auc_min,train_auc_avg,train_auc_max,val_loss_min,val_loss_avg,val_loss_max,val_acc_min,val_acc_avg,val_acc_max,val_auc_min,val_auc_avg,val_auc_max
0,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
1,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
2,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
3,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
4,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
5,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
6,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
7,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
8,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79
9,0.44,0.47,0.61,69.3,80.15,81.28,0.69,0.8,0.81,0.47,0.5,0.59,73.68,78.86,79.3,0.74,0.79,0.79


In [130]:
start = 0
end = 50

print(f"{str(start + 1).rjust(3, '0')}")

001


In [132]:
output_df.to_csv(os.path.join(DATASET_PATH, "experiments/min-avg-max", 
                              f"batch-{str(start + 1).rjust(3, '0')}-{str(end).rjust(3, '0')}.csv"), 
                 header=True, index=False)

# Whole Workflow

In [143]:
start = 0
end = 5

output_rows = []

for item in perm_lst[start:end]:
    # Generate histone item name
    item_name = '-'.join(item)

    print(f"Item: {item_name}")

    # Create dataframe based on histone marker permutation item
    perm_histone = E066_pl.with_columns(
        pl.struct(item).map_elements(
            lambda x: [x[col_name] for col_name in item],
            return_dtype = pl.List(pl.List(pl.Float64))
        )
        .alias('histone')
    )

    # Select X and y column
    X = perm_histone.select(pl.col('histone')).to_series().to_list()
    y = perm_histone.select(pl.col('label')).to_series().to_list()

    # Convert to Numpy array
    X = np.array(X)
    y = np.array(y)

    # Split the dataset into training, validation, and test sets
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.666, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Convert numpy arrays to PyTorch tensors
    X_train = torch.from_numpy(X_train).float().unsqueeze(1).to(device)
    X_val = torch.from_numpy(X_val).float().unsqueeze(1).to(device)
    X_test = torch.from_numpy(X_test).float().unsqueeze(1).to(device)

    y_train = torch.from_numpy(y_train).float().to(device)
    y_val = torch.from_numpy(y_val).float().to(device)
    y_test = torch.from_numpy(y_test).float().to(device)

    # Create DataLoaders
    batch_size = 32

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    val_dataset = TensorDataset(X_val, y_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)


    # Define Deep Learning model instannce
    model = DeepClassifier(input_dim = [len(item), 100]).to(device)
    criterion = nn.NLLLoss()
    # optimizer = optim.Adam(model.parameters(), lr=0.001)
    optimizer = optim.SGD(model.parameters(), lr=0.001)
    num_epochs = 100

    # For tracking train and validation
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    train_aucs, val_aucs = [], []

    # For saving the best model
    best_val_metric = float('-inf')
    best_model_path = os.path.join(DATASET_PATH, "experiments", "model", f'{item_name}.pth')

    for epoch in range(num_epochs):
        # Training phase
        model.train()  # Set the model to training mode
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        train_true_labels = []
        train_predicted_probs = []
        
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.cuda(), labels.type(torch.LongTensor).cuda())
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            train_true_labels.extend(labels.cpu().numpy())
            train_predicted_probs.extend(predicted.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = 100 * train_correct / train_total
        train_auc_score = roc_auc_score(train_true_labels, train_predicted_probs)
        
        train_losses.append(round(avg_train_loss, 4))
        train_accuracies.append(round(train_accuracy, 2))
        train_aucs.append(round(train_auc_score, 2))
        
        # Validation phase
        model.eval()  # Set the model to evaluation mode
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_metric = 0
        val_true_labels = []
        val_predicted_probs = []
        
        # Training and Validation
        with torch.no_grad():  # Disable gradient computation
            for inputs, labels in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs.cuda(), labels.type(torch.LongTensor).cuda())
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                val_true_labels.extend(labels.cpu().numpy())
                val_predicted_probs.extend(predicted.cpu().numpy())

            avg_val_loss = val_loss / len(val_loader)
            val_accuracy = 100 * val_correct / val_total
            val_metric = val_correct / len(val_loader)

            # Save the best model
            if val_metric > best_val_metric:
                best_val_metric = val_metric
                torch.save(model.state_dict(), best_model_path)
                # print(f"New best model saved with validation metric: {best_val_metric:.4f}")
            
            val_true_labels = np.array(val_true_labels)
            val_predicted_probs = np.array(val_predicted_probs)
            val_auc_score = roc_auc_score(val_true_labels, val_predicted_probs)

            val_losses.append(round(avg_val_loss, 4))
            val_accuracies.append(round(val_accuracy, 2))
            val_aucs.append(round(val_auc_score, 2))
            
            if ((epoch + 1) % 10 == 0):    
                print(f'Epoch [{epoch+1}/{num_epochs}] - '
                    f'[TRAIN] Loss: {avg_train_loss:.4f}, Accuracy: {train_accuracy:.2f}%, AUC: {train_auc_score:.2f} - '
                    f'[VAL] Loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.2f}%, AUC: {val_auc_score:.2f}')
    
    print("Training finished!")

    # Load the best model
    best_model = DeepClassifier(input_dim = [len(item), 100]).to(device)
    best_model.load_state_dict(torch.load(best_model_path))
    best_model.eval()

    # Predict on test set
    test_predictions = []
    test_true_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = best_model(inputs)
            # probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            test_predictions.extend(predicted.cpu().numpy())
            test_true_labels.extend(labels.cpu().numpy())

    # Calculate evaluation metric
    test_acc_score = round(accuracy_score(test_true_labels, test_predictions) * 100, 2)
    print(f"Accuracy Score on test set: {test_acc_score:.2f} %")
    test_auc_score = round(roc_auc_score(test_true_labels, test_predictions), 2)
    print(f"AUC Score on test set: {test_auc_score:.2f}\n")

    # Create dataframe from array
    experiment_results = {
        'train_loss': train_losses,
        'train_accuracy': train_accuracies,
        'train_auc': train_aucs,
        'val_loss': val_losses,
        'val_accuracy': val_accuracies,
        'val_auc': val_aucs
    }

    # Saving the experiments into CSV files
    experiment_df = pl.DataFrame(experiment_results)
    experiment_df.write_csv(os.path.join(DATASET_PATH, 'experiments', f'{item_name}.csv'))

    # Find the min, avg, max for training and validation step
    train_loss_min, train_loss_avg, train_loss_max = min_avg_max(train_losses)
    train_acc_min, train_acc_avg, train_acc_max = min_avg_max(train_accuracies)
    train_auc_min, train_auc_avg, train_auc_max = min_avg_max(train_aucs)

    val_loss_min, val_loss_avg, val_loss_max = min_avg_max(val_losses)
    val_acc_min, val_acc_avg, val_acc_max = min_avg_max(val_accuracies)
    val_auc_min, val_auc_avg, val_auc_max = min_avg_max(val_aucs)

    output_rows.append([
        item_name, 
        train_loss_min, train_loss_avg, train_loss_max,
        train_acc_min, train_acc_avg, train_acc_max,
        train_auc_min, train_auc_avg, train_auc_max,
        val_loss_min, val_loss_avg, val_loss_max,
        val_acc_min, val_acc_avg, val_acc_max,
        val_auc_min, val_auc_avg, val_auc_max,
    ])


# Saving current batch results
output_rows_df = pd.DataFrame(output_rows, 
    columns=[
        "item_name",
        "train_loss_min", "train_loss_avg", "train_loss_max",
        "train_acc_min", "train_acc_avg", "train_acc_max",
        "train_auc_min", "train_auc_avg", "train_auc_max",
        "val_loss_min", "val_loss_avg", "val_loss_max",
        "val_acc_min", "val_acc_avg", "val_acc_max",
        "val_auc_min", "val_auc_avg", "val_auc_max",
    ]
)

print("Saving the final results")
output_rows_df.to_csv(os.path.join(DATASET_PATH, "experiments/min-avg-max", 
                              f"{str(start + 1).rjust(3, '0')}-{str(end).rjust(3, '0')}.csv"), 
                 header=True, index=False)

print("Batch FINISHED!!!")

Item: H3K4me3
Epoch [10/100] - [TRAIN] Loss: 0.5400, Accuracy: 78.14%, AUC: 0.78 - [VAL] Loss: 0.5444, Accuracy: 77.79%, AUC: 0.78
Epoch [20/100] - [TRAIN] Loss: 0.5068, Accuracy: 78.54%, AUC: 0.79 - [VAL] Loss: 0.5175, Accuracy: 77.21%, AUC: 0.77
Epoch [30/100] - [TRAIN] Loss: 0.4981, Accuracy: 78.46%, AUC: 0.78 - [VAL] Loss: 0.5092, Accuracy: 77.42%, AUC: 0.77
Epoch [40/100] - [TRAIN] Loss: 0.4919, Accuracy: 78.81%, AUC: 0.79 - [VAL] Loss: 0.5086, Accuracy: 77.16%, AUC: 0.77
Epoch [50/100] - [TRAIN] Loss: 0.4988, Accuracy: 79.21%, AUC: 0.79 - [VAL] Loss: 0.5079, Accuracy: 77.16%, AUC: 0.77
Epoch [60/100] - [TRAIN] Loss: 0.4866, Accuracy: 79.12%, AUC: 0.79 - [VAL] Loss: 0.5070, Accuracy: 77.19%, AUC: 0.77
Epoch [70/100] - [TRAIN] Loss: 0.4861, Accuracy: 79.30%, AUC: 0.79 - [VAL] Loss: 0.5129, Accuracy: 76.86%, AUC: 0.77
Epoch [80/100] - [TRAIN] Loss: 0.4852, Accuracy: 79.58%, AUC: 0.80 - [VAL] Loss: 0.5148, Accuracy: 76.93%, AUC: 0.77
Epoch [90/100] - [TRAIN] Loss: 0.4865, Accuracy: 7

In [144]:
item = [1, 2, 3]